Metodologia obtida em https://www.kaggle.com/code/aaryandahalnp/predict-fertilizer-xgboost

### Bibliotecas

In [49]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.preprocessing import LabelEncoder

import warnings
import os

### Conficurações iniciais

In [50]:
# Suprimir avisos e definir opções de exibição
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Definir diretório
os.chdir("G:\\Meu Drive\\MeuDrive2\\academico\\3.kaggle\\3.Predicting_Optimal_Fertilizers")

### EDA

In [51]:
# Importando os dados
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample_submission = pd.read_csv("sample_submission.csv")

In [52]:
# Primeiras observações
train.head()

,id,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous,Fertilizer Name
0,0,37,70,36,Clayey,Sugarcane,36,4,5,28-28
1,1,27,69,65,Sandy,Millets,30,6,18,28-28
2,2,29,63,32,Sandy,Millets,24,12,16,17-17-17
3,3,35,62,54,Sandy,Barley,39,12,4,10-26-26
4,4,35,58,43,Red,Paddy,37,2,16,DAP


In [53]:
# Renomeando a coluna Temparature para temperatur
train.rename(columns = {'Temparature': 'Temperature'}, inplace=True)
test.rename(columns = {'Temparature': 'Temperature'}, inplace=True)

### Preparação

In [54]:
# Separando em treino e teste
target = "Fertilizer Name"
X = train.drop(columns=[target, 'id'])
y = train[target]

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)

In [55]:
# Colunas numéricas
numeric_cols = X_train.select_dtypes(include=np.number).columns.tolist()

# Colunas categóricas
categorical_cols = X_train.select_dtypes(include='object').columns.tolist()

In [56]:
# Criando dummies
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='first')
encoder.fit(X_train[categorical_cols])

OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)

In [57]:
# Novos nomes das colunas dummy
encoded_cols = encoder.get_feature_names_out(categorical_cols)

In [58]:
# Df's apenas com os novos atributos criados - X_train
X_train_encoded = pd.DataFrame(
    encoder.transform(X_train[categorical_cols]),
    columns=encoded_cols,
    index=X_train.index
)

# Df's apenas com os novos atributos criados - X_valid
X_valid_encoded = pd.DataFrame(
    encoder.transform(X_valid[categorical_cols]),
    columns=encoded_cols,
    index=X_valid.index
)

In [59]:
# Df's finais com os atributos numéricos e as variáveis dummies
X_train_final = pd.concat([X_train[numeric_cols], X_train_encoded], axis=1)
X_valid_final = pd.concat([X_valid[numeric_cols], X_valid_encoded], axis=1)

In [60]:
# Padronizando os dados
scaler = MinMaxScaler().fit(X_train_final[numeric_cols])  # Ajusta apenas no treino

# Substitui colunas numéricas pelas normalizadas
X_train_final[numeric_cols] = scaler.transform(X_train_final[numeric_cols])
X_valid_final[numeric_cols] = scaler.transform(X_valid_final[numeric_cols])

### Codificação

In [61]:
# Convertendo os rótulos
# Cria o codificador de rótulos
le = LabelEncoder()

# Ajusta e transforma o y_train (com os rótulos originais)
y_train_enc = le.fit_transform(y_train)

# Apenas transforma y_valid (usando o mesmo mapeamento do treino)
y_valid_enc = le.transform(y_valid)

In [62]:
def make_predictions(inputs, model):
    probs = model.predict_proba(inputs)
    top3_idx = np.argsort(probs, axis=1)[:, -3:][:, ::-1]
    predictions = le.inverse_transform(top3_idx.ravel()).reshape(top3_idx.shape)
    return predictions

In [63]:
def accuracy(actual, predicted, k=3):
    if len(actual) != len(predicted):
        raise ValueError(f"Length mismatch: actual has {len(actual)}, predicted has {len(predicted)}")

    score = 0.0
    for a, p in zip(actual, predicted):
        if a in p[:k]:
            rank = p.index(a)
            score += 1.0 / (rank + 1)
        # If not in top-k, score += 0
    return score / len(actual)

In [64]:
def probability(actual, predictions):
    actual_labels = le.inverse_transform(actual)
    actual_labels = actual_labels.tolist()
    predictions = predictions.tolist()
    score = accuracy(actual_labels, predictions, k=3)
    print(f"Score: {score:.5f}")

### XGB

In [65]:
"""from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

# Definir o classificador base
xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss')

param_grid = {
    'n_estimators': [100, 200, 300, 400],   # mais granularidade
    'max_depth': [3, 5, 7],                 # árvores mais rasas e intermediárias
    'learning_rate': [0.05, 0.1, 0.2, 0.3]  # taxas suaves a mais agressivas
}

# Busca com validação cruzada
grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring='accuracy',  # ou outra métrica, como make_scorer(mapk_scorer) para MAP@3
    cv=5,
    verbose=3
)

# Executar a busca
grid_search.fit(X_train_final_filtered, y_train_enc)

# Modelo com os melhores parâmetros
best_model_xgb = grid_search.best_estimator_
# Melhores parâmetros: {'learning_rate': 0.3, 'max_depth': 5, 'n_estimators': 400}

# Visualizar os melhores parâmetros encontrados
print("Melhores parâmetros:", grid_search.best_params_)
print(f"Melhor score médio na validação cruzada: {grid_search.best_score_:.4f}")

val_predictions = make_predictions(X_valid_final_filtered, best_model_xgb)
probability(y_valid_enc, val_predictions)"""

'from sklearn.model_selection import GridSearchCV\nfrom xgboost import XGBClassifier\n\n# Definir o classificador base\nxgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric=\'mlogloss\')\n\nparam_grid = {\n    \'n_estimators\': [100, 200, 300, 400],   # mais granularidade\n    \'max_depth\': [3, 5, 7],                 # árvores mais rasas e intermediárias\n    \'learning_rate\': [0.05, 0.1, 0.2, 0.3]  # taxas suaves a mais agressivas\n}\n\n# Busca com validação cruzada\ngrid_search = GridSearchCV(\n    estimator=xgb,\n    param_grid=param_grid,\n    scoring=\'accuracy\',  # ou outra métrica, como make_scorer(mapk_scorer) para MAP@3\n    cv=5,\n    verbose=3\n)\n\n# Executar a busca\ngrid_search.fit(X_train_final_filtered, y_train_enc)\n\n# Modelo com os melhores parâmetros\nbest_model_xgb = grid_search.best_estimator_\n# Melhores parâmetros: {\'learning_rate\': 0.3, \'max_depth\': 5, \'n_estimators\': 400}\n\n# Visualizar os melhores parâmetros encontrados\nprint("

In [66]:
from xgboost import XGBClassifier

# Reutilizar os melhores parâmetros encontrados
final_model = XGBClassifier(
    learning_rate=0.3,
    max_depth=5,
    n_estimators=400,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42
)

# Treinar o modelo com os dados de treino
final_model.fit(X_train_final, y_train_enc)

# Fazer predições no conjunto de validação
val_predictions = make_predictions(X_valid_final, final_model)

# Avaliar a performance com a função personalizada
probability(y_valid_enc, val_predictions)

Score: 0.33780


In [40]:
"""from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

# Definir o classificador base
dt = DecisionTreeClassifier(random_state=42)

# Grade de parâmetros simplificada
param_grid = {
    'max_depth': [3, 5],              # árvores rasas
    'min_samples_split': [2],         # mínimo padrão
    'min_samples_leaf': [1]           # padrão
}

# Busca com validação cruzada
grid_search = GridSearchCV(
    estimator=dt,
    param_grid=param_grid,
    scoring='accuracy',  # ou outra métrica como make_scorer(mapk_scorer)
    cv=5,
    verbose=3
)

# Executar a busca
grid_search.fit(X_train_final, y_train_enc)

# Modelo com os melhores parâmetros
best_model_dt = grid_search.best_estimator_

# Visualizar os melhores parâmetros encontrados
print("Melhores parâmetros:", grid_search.best_params_)
print(f"Melhor score médio na validação cruzada: {grid_search.best_score_:.4f}")

# Avaliação final no conjunto de validação
val_predictions = make_predictions(X_valid_final, best_model_dt)
probability(y_valid_enc, val_predictions)"""

'from sklearn.tree import DecisionTreeClassifier\nfrom sklearn.model_selection import GridSearchCV\n\n# Definir o classificador base\ndt = DecisionTreeClassifier(random_state=42)\n\n# Grade de parâmetros simplificada\nparam_grid = {\n    \'max_depth\': [3, 5],              # árvores rasas\n    \'min_samples_split\': [2],         # mínimo padrão\n    \'min_samples_leaf\': [1]           # padrão\n}\n\n# Busca com validação cruzada\ngrid_search = GridSearchCV(\n    estimator=dt,\n    param_grid=param_grid,\n    scoring=\'accuracy\',  # ou outra métrica como make_scorer(mapk_scorer)\n    cv=5,\n    verbose=3\n)\n\n# Executar a busca\ngrid_search.fit(X_train_final, y_train_enc)\n\n# Modelo com os melhores parâmetros\nbest_model_dt = grid_search.best_estimator_\n\n# Visualizar os melhores parâmetros encontrados\nprint("Melhores parâmetros:", grid_search.best_params_)\nprint(f"Melhor score médio na validação cruzada: {grid_search.best_score_:.4f}")\n\n# Avaliação final no conjunto de validaçã

In [41]:
"""from lightgbm import LGBMClassifier
from sklearn.model_selection import GridSearchCV

# Definir o classificador base
lgbm = LGBMClassifier(random_state=42)

# Grade de parâmetros simplificada
param_grid = {
    'n_estimators': [100, 300],   # número de árvores
    'max_depth': [3, 5],          # profundidade máxima
    'learning_rate': [0.1]        # taxa de aprendizado padrão
}

# Busca com validação cruzada
grid_search = GridSearchCV(
    estimator=lgbm,
    param_grid=param_grid,
    scoring='accuracy',  # ou make_scorer(mapk_scorer)
    cv=5,
    verbose=3
)

# Executar a busca
grid_search.fit(X_train_final, y_train_enc)

# Modelo com os melhores parâmetros
best_model_lgbm = grid_search.best_estimator_
# Melhores parâmetros: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 300}

# Visualizar os melhores parâmetros encontrados
print("Melhores parâmetros:", grid_search.best_params_)
print(f"Melhor score médio na validação cruzada: {grid_search.best_score_:.4f}")

# Avaliação final no conjunto de validação
val_predictions = make_predictions(X_valid_final, best_model_lgbm)
probability(y_valid_enc, val_predictions)"""

'from lightgbm import LGBMClassifier\nfrom sklearn.model_selection import GridSearchCV\n\n# Definir o classificador base\nlgbm = LGBMClassifier(random_state=42)\n\n# Grade de parâmetros simplificada\nparam_grid = {\n    \'n_estimators\': [100, 300],   # número de árvores\n    \'max_depth\': [3, 5],          # profundidade máxima\n    \'learning_rate\': [0.1]        # taxa de aprendizado padrão\n}\n\n# Busca com validação cruzada\ngrid_search = GridSearchCV(\n    estimator=lgbm,\n    param_grid=param_grid,\n    scoring=\'accuracy\',  # ou make_scorer(mapk_scorer)\n    cv=5,\n    verbose=3\n)\n\n# Executar a busca\ngrid_search.fit(X_train_final, y_train_enc)\n\n# Modelo com os melhores parâmetros\nbest_model_lgbm = grid_search.best_estimator_\n# Melhores parâmetros: {\'learning_rate\': 0.1, \'max_depth\': 5, \'n_estimators\': 300}\n\n# Visualizar os melhores parâmetros encontrados\nprint("Melhores parâmetros:", grid_search.best_params_)\nprint(f"Melhor score médio na validação cruzada: {

In [42]:
"""from catboost import CatBoostClassifier
from sklearn.model_selection import GridSearchCV

# Definir o classificador base
catboost = CatBoostClassifier(
    random_state=42,
    verbose=0,           # evita saída extensa do CatBoost
    loss_function='MultiClass'
)

# Grade de parâmetros simplificada
param_grid = {
    'iterations': [100, 300],     # número de árvores
    'depth': [3, 5],              # profundidade da árvore
    'learning_rate': [0.1]        # taxa de aprendizado
}

# Busca com validação cruzada
grid_search = GridSearchCV(
    estimator=catboost,
    param_grid=param_grid,
    scoring='accuracy',  # ou make_scorer(mapk_scorer)
    cv=5,
    verbose=3
)

# Executar a busca
grid_search.fit(X_train_final, y_train_enc)

# Modelo com os melhores parâmetros
best_model_catboost = grid_search.best_estimator_
# Melhores parâmetros: {'depth': 5, 'iterations': 300, 'learning_rate': 0.1}

# Visualizar os melhores parâmetros encontrados
print("Melhores parâmetros:", grid_search.best_params_)
print(f"Melhor score médio na validação cruzada: {grid_search.best_score_:.4f}")

# Avaliação final no conjunto de validação
val_predictions = make_predictions(X_valid_final, best_model_catboost)
probability(y_valid_enc, val_predictions)"""

'from catboost import CatBoostClassifier\nfrom sklearn.model_selection import GridSearchCV\n\n# Definir o classificador base\ncatboost = CatBoostClassifier(\n    random_state=42,\n    verbose=0,           # evita saída extensa do CatBoost\n    loss_function=\'MultiClass\'\n)\n\n# Grade de parâmetros simplificada\nparam_grid = {\n    \'iterations\': [100, 300],     # número de árvores\n    \'depth\': [3, 5],              # profundidade da árvore\n    \'learning_rate\': [0.1]        # taxa de aprendizado\n}\n\n# Busca com validação cruzada\ngrid_search = GridSearchCV(\n    estimator=catboost,\n    param_grid=param_grid,\n    scoring=\'accuracy\',  # ou make_scorer(mapk_scorer)\n    cv=5,\n    verbose=3\n)\n\n# Executar a busca\ngrid_search.fit(X_train_final, y_train_enc)\n\n# Modelo com os melhores parâmetros\nbest_model_catboost = grid_search.best_estimator_\n# Melhores parâmetros: {\'depth\': 5, \'iterations\': 300, \'learning_rate\': 0.1}\n\n# Visualizar os melhores parâmetros encontr

In [48]:
"""from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

# Modelos base já otimizados
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Modelos base
xgb_best = XGBClassifier(
    learning_rate=0.3,
    max_depth=5,
    n_estimators=400,
    verbose=3,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42
)

lgbm_best = LGBMClassifier(
    learning_rate=0.1,
    max_depth=5,
    n_estimators=300,
    random_state=42,
    verbose=3
)

catboost_best = CatBoostClassifier(
    depth=5,
    iterations=300,
    learning_rate=0.1,
    random_state=42,
    verbose=3,
    loss_function='MultiClass'
)

# Empilhamento com Logistic Regression como meta-modelo
stacked_model = StackingClassifier(
    estimators=[
        ('xgb', xgb_best),
        ('lgbm', lgbm_best),
        ('catboost', catboost_best)
    ],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5,
    passthrough=False,  # pode colocar True se quiser empilhar também as features originais
    verbose=3
)

# Treinamento do modelo empilhado
stacked_model.fit(X_train_final, y_train_enc)

# Previsões no conjunto de validação
val_predictions = make_predictions(X_valid_final, stacked_model)

# Avaliação final
probability(y_valid_enc, val_predictions)"""

"from sklearn.ensemble import StackingClassifier\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.preprocessing import LabelEncoder\n\n# Modelos base já otimizados\nfrom xgboost import XGBClassifier\nfrom lightgbm import LGBMClassifier\nfrom catboost import CatBoostClassifier\n\n# Modelos base\nxgb_best = XGBClassifier(\n    learning_rate=0.3,\n    max_depth=5,\n    n_estimators=400,\n    verbose=3,\n    use_label_encoder=False,\n    eval_metric='mlogloss',\n    random_state=42\n)\n\nlgbm_best = LGBMClassifier(\n    learning_rate=0.1,\n    max_depth=5,\n    n_estimators=300,\n    random_state=42,\n    verbose=3\n)\n\ncatboost_best = CatBoostClassifier(\n    depth=5,\n    iterations=300,\n    learning_rate=0.1,\n    random_state=42,\n    verbose=3,\n    loss_function='MultiClass'\n)\n\n# Empilhamento com Logistic Regression como meta-modelo\nstacked_model = StackingClassifier(\n    estimators=[\n        ('xgb', xgb_best),\n        ('lgbm', lgbm_best),\n        ('catboo

### Teste

In [67]:
# --- 9. Preparar test ---
test.rename(columns={'Temparature': 'Temperature'}, inplace=True)
X_test = test.drop(columns=['id'])

# --- 10. Transformar test (encoder + scaler já treinados) ---
X_test_encoded = pd.DataFrame(encoder.transform(X_test[categorical_cols]), columns=encoded_cols, index=X_test.index)
X_test_final = pd.concat([X_test[numeric_cols], X_test_encoded], axis=1)
X_test_final[numeric_cols] = scaler.transform(X_test_final[numeric_cols])

In [68]:
# --- 11. Predizer Top-3 fertilizantes ---
probs = final_model.predict_proba(X_test_final)
top3_idx = np.argsort(probs, axis=1)[:, -3:][:, ::-1]  # top-3 em ordem decrescente
top3_labels = le.inverse_transform(top3_idx.ravel()).reshape(top3_idx.shape)
top3_strings = [' '.join(row) for row in top3_labels]

### Submissão

In [69]:
# --- 12. Criar submissão ---
submission = sample_submission.copy()
submission["Fertilizer Name"] = top3_strings
submission.to_csv("xgboost.csv", index=False)